<a href="https://colab.research.google.com/github/HariNiveditha/IndustryProjectPrediction/blob/main/Dataset_extraction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
 !pip install pdfplumber -q

 from google.colab import files
 uploaded = files.upload()

import zipfile
with zipfile.ZipFile("dataset_pdfs.zip", "r") as z:   # add .zip here
    z.extractall("dataset_pdfs")

Saving dataset_pdfs.zip to dataset_pdfs (1).zip


In [ ]:
import pdfplumber
import csv
import re
import glob
import os

INPUT_DIR = "dataset_pdfs"
OUT_PATH = "dataset_pdfs/all_ongoing_projects_combined.csv"

HEADER = [
    "report_file", "sl_no", "project_name", "agency", "project_code", "state",
    "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

def split_two(cell):
    if not cell:
        return "", ""
    parts = cell.split("\n")
    main = parts[0].strip()
    rev = parts[1].strip("() ") if len(parts) > 1 else ""
    return main, rev

def parse_project_cell(cell):
    if not cell:
        return "", "", ""
    text = cell.replace("\n", " ").strip()
    code_match = re.search(r"\((\d{5,7})\)\s*$", text)
    code = code_match.group(1) if code_match else ""
    if code_match:
        text = text[:code_match.start()].strip()
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)
    agency = agency_match.group(1).strip() if agency_match else ""
    if agency_match:
        text = text[:agency_match.start()].strip()
    return text, agency, code

def find_target_pages(pdf):
    """Auto-detect pages whose title is 'All Ongoing Projects' (not the
    North-East-only or Completed/Newly-Added variants)."""
    pages = []
    for i, page in enumerate(pdf.pages):
        txt = page.extract_text() or ""
        first_line = txt.split("\n")[0].strip() if txt else ""
        if first_line == "All Ongoing Projects":
            pages.append(i)
    return pages

def extract_pdf(path):
    rows_out = []
    fname = os.path.basename(path)
    current_ministry = ""
    current_sector = ""

    with pdfplumber.open(path) as pdf:
        target_pages = find_target_pages(pdf)
        if not target_pages:
            print(f"  WARNING: no 'All Ongoing Projects' pages found in {fname} — check its layout manually")
            return rows_out

        for pnum in target_pages:
            page = pdf.pages[pnum]
            tables = page.extract_tables()
            data_table = None
            for t in tables:
                if t and t[0] and t[0][0] == "Sl.No":
                    data_table = t
                    break
            if not data_table:
                continue

            for row in data_table[1:]:
                if not row or all(c in (None, "") for c in row):
                    continue
                sl_no = (row[0] or "").strip()

                if not sl_no and row[1] and all(c in (None, "") for c in row[2:]):
                    label = row[1].strip()
                    if label.lower().startswith(("ministry", "department")):
                        current_ministry = label
                        current_sector = ""
                    else:
                        current_sector = label
                    continue

                if not sl_no.isdigit():
                    continue

                project_name, agency, code = parse_project_cell(row[1])
                state = re.sub(r"\s*\n\s*", " ", (row[2] or "")).strip()
                approval, revised_start = split_two(row[3])
                target_doc, revised_doc = split_two(row[4])
                orig_cost, revised_cost = split_two(row[5])
                expenditure = (row[6] or "").strip()
                progress = (row[7] or "").strip()

                rows_out.append([
                    fname, sl_no, project_name, agency, code, state,
                    current_ministry, current_sector,
                    approval, revised_start,
                    target_doc, revised_doc,
                    orig_cost, revised_cost,
                    expenditure, progress,
                ])
    return rows_out

def main():
    pdf_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdf")))
    print(f"Found {len(pdf_files)} PDFs")

    all_rows = []
    for path in pdf_files:
        print(f"Processing {os.path.basename(path)} ...")
        rows = extract_pdf(path)
        print(f"  -> {len(rows)} rows")
        all_rows.extend(rows)

    with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(all_rows)

    print(f"\nTotal rows: {len(all_rows)} -> {OUT_PATH}")

main()

Found 0 PDFs

Total rows: 0 -> dataset_pdfs/all_ongoing_projects_combined.csv


In [ ]:
import os
for root, dirs, filenames in os.walk("dataset_pdfs"):
    for f in filenames:
        print(os.path.join(root, f))

dataset_pdfs/all_ongoing_projects_combined.csv
dataset_pdfs/dataset_pdfs/FR_May2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_April2026.pdf
dataset_pdfs/dataset_pdfs/FlashReport_February_2026.pdf
dataset_pdfs/dataset_pdfs/FlashReport_October_2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_August_2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_July_2026.pdf
dataset_pdfs/dataset_pdfs/FlashReport_November_2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_March_2026.pdf
dataset_pdfs/dataset_pdfs/FlashReport_June_2026.pdf
dataset_pdfs/dataset_pdfs/FRApril2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_January_2026.pdf
dataset_pdfs/dataset_pdfs/FR_JUNE_2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_December_2025.pdf
dataset_pdfs/dataset_pdfs/FlashReport_May2026.pdf
dataset_pdfs/dataset_pdfs/FlashReport_July_2025.pdf


In [ ]:
import pdfplumber
import csv
import re
import glob
import os

INPUT_DIR = "dataset_pdfs/dataset_pdfs"
OUT_PATH = "dataset_pdfs/all_ongoing_projects_combined.csv"

HEADER = [
    "report_file", "sl_no", "project_name", "agency", "project_code", "state",
    "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

def split_two(cell):
    if not cell:
        return "", ""
    parts = cell.split("\n")
    main = parts[0].strip()
    rev = parts[1].strip("() ") if len(parts) > 1 else ""
    return main, rev

def parse_project_cell(cell):
    if not cell:
        return "", "", ""
    text = cell.replace("\n", " ").strip()
    code_match = re.search(r"\((\d{5,7})\)\s*$", text)
    code = code_match.group(1) if code_match else ""
    if code_match:
        text = text[:code_match.start()].strip()
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)
    agency = agency_match.group(1).strip() if agency_match else ""
    if agency_match:
        text = text[:agency_match.start()].strip()
    return text, agency, code

def find_target_pages(pdf):
    """Auto-detect pages whose title is 'All Ongoing Projects' (not the
    North-East-only or Completed/Newly-Added variants)."""
    pages = []
    for i, page in enumerate(pdf.pages):
        txt = page.extract_text() or ""
        first_line = txt.split("\n")[0].strip() if txt else ""
        if first_line == "All Ongoing Projects":
            pages.append(i)
    return pages

def extract_pdf(path):
    rows_out = []
    fname = os.path.basename(path)
    current_ministry = ""
    current_sector = ""

    with pdfplumber.open(path) as pdf:
        target_pages = find_target_pages(pdf)
        if not target_pages:
            print(f"  WARNING: no 'All Ongoing Projects' pages found in {fname} — check its layout manually")
            return rows_out

        for pnum in target_pages:
            page = pdf.pages[pnum]
            tables = page.extract_tables()
            data_table = None
            for t in tables:
                if t and t[0] and t[0][0] == "Sl.No":
                    data_table = t
                    break
            if not data_table:
                continue

            for row in data_table[1:]:
                if not row or all(c in (None, "") for c in row):
                    continue
                sl_no = (row[0] or "").strip()

                if not sl_no and row[1] and all(c in (None, "") for c in row[2:]):
                    label = row[1].strip()
                    if label.lower().startswith(("ministry", "department")):
                        current_ministry = label
                        current_sector = ""
                    else:
                        current_sector = label
                    continue

                if not sl_no.isdigit():
                    continue

                project_name, agency, code = parse_project_cell(row[1])
                state = re.sub(r"\s*\n\s*", " ", (row[2] or "")).strip()
                approval, revised_start = split_two(row[3])
                target_doc, revised_doc = split_two(row[4])
                orig_cost, revised_cost = split_two(row[5])
                expenditure = (row[6] or "").strip()
                progress = (row[7] or "").strip()

                rows_out.append([
                    fname, sl_no, project_name, agency, code, state,
                    current_ministry, current_sector,
                    approval, revised_start,
                    target_doc, revised_doc,
                    orig_cost, revised_cost,
                    expenditure, progress,
                ])
    return rows_out

def main():
    pdf_files = sorted(glob.glob(os.path.join(INPUT_DIR, "*.pdf")))
    print(f"Found {len(pdf_files)} PDFs")

    all_rows = []
    for path in pdf_files:
        print(f"Processing {os.path.basename(path)} ...")
        rows = extract_pdf(path)
        print(f"  -> {len(rows)} rows")
        all_rows.extend(rows)

    with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(all_rows)

    print(f"\nTotal rows: {len(all_rows)} -> {OUT_PATH}")

main()

Found 15 PDFs
Processing FRApril2025.pdf ...
  -> 0 rows
Processing FR_JUNE_2025.pdf ...
  -> 0 rows
Processing FR_May2025.pdf ...
  -> 0 rows
Processing FlashReport_April2026.pdf ...
  -> 1981 rows
Processing FlashReport_August_2025.pdf ...
  -> 800 rows
Processing FlashReport_December_2025.pdf ...
  -> 1345 rows
Processing FlashReport_February_2026.pdf ...
  -> 1897 rows
Processing FlashReport_January_2026.pdf ...
  -> 1605 rows
Processing FlashReport_July_2025.pdf ...
  -> 714 rows
Processing FlashReport_July_2026.pdf ...
  -> 1775 rows
Processing FlashReport_June_2026.pdf ...
  -> 1847 rows
Processing FlashReport_March_2026.pdf ...
  -> 1869 rows
Processing FlashReport_May2026.pdf ...
  -> 1987 rows
Processing FlashReport_November_2025.pdf ...
  -> 823 rows
Processing FlashReport_October_2025.pdf ...
  -> 798 rows

Total rows: 17441 -> dataset_pdfs/all_ongoing_projects_combined.csv


In [ ]:
import pdfplumber

with pdfplumber.open("dataset_pdfs/dataset_pdfs/FRApril2025.pdf") as pdf:
    for i, page in enumerate(pdf.pages):
        txt = page.extract_text() or ""
        if "Sl.No" in txt and "Project Name" in txt:
            print(i+1, "|", txt.split("\n")[0])

In [ ]:
import pdfplumber
import csv
import re
import os

INPUT_DIR = "dataset_pdfs/dataset_pdfs"
OUT_PATH = "dataset_pdfs/old_format_3files.csv"

TARGET_FILES = ["FRApril2025.pdf", "FR_JUNE_2025.pdf", "FR_May2025.pdf"]

HEADER = [
    "report_file", "format", "sl_no", "project_name", "agency", "project_code", "state",
    "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

INDIAN_STATES_UTS = [
    "ANDAMAN AND NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM",
    "BIHAR", "CHANDIGARH", "CHHATTISGARH",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "DELHI", "GOA", "GUJARAT",
    "HARYANA", "HIMACHAL PRADESH", "JAMMU AND KASHMIR", "JHARKHAND",
    "KARNATAKA", "KERALA", "LADAKH", "LAKSHADWEEP", "MADHYA PRADESH",
    "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA",
    "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM", "TAMIL NADU", "TELANGANA",
    "TRIPURA", "UTTAR PRADESH", "UTTARAKHAND", "WEST BENGAL",
]

def parse_project_cell(cell):
    if not cell:
        return "", "", ""
    text = cell.replace("\n", " ").strip()
    code_match = re.search(r"\((\w{5,10})\s*\)\s*$", text)
    code = code_match.group(1) if code_match else ""
    if code_match:
        text = text[:code_match.start()].strip()
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)
    agency = agency_match.group(1).strip() if agency_match else ""
    if agency_match:
        text = text[:agency_match.start()].strip()
    return text, agency, code

def normalize_state(raw):
    if not raw:
        return raw
    raw_clean = raw.strip().upper()
    if raw_clean in INDIAN_STATES_UTS:
        return raw_clean
    for full in INDIAN_STATES_UTS:
        if full.startswith(raw_clean):
            return full
    return raw_clean

COLUMNS = [
    ("state",      0,   70),
    ("sector",     70,  127),
    ("sl_no",      127, 150),
    ("name",       150, 278),
    ("approval",   278, 333),
    ("commission", 333, 415),
    ("cost",       415, 497),
    ("exp",        497, 560),
    ("progress",   560, 650),
]

def col_for_x(x0):
    for name, lo, hi in COLUMNS:
        if lo <= x0 < hi:
            return name
    return None

def group_lines(words, tol=3):
    words = sorted(words, key=lambda w: w["top"])
    lines, current, current_top = [], [], None
    for w in words:
        if current_top is None or abs(w["top"] - current_top) <= tol:
            current.append(w)
            current_top = w["top"] if current_top is None else current_top
        else:
            lines.append(current)
            current, current_top = [w], w["top"]
    if current:
        lines.append(current)
    out = []
    for line in lines:
        line_sorted = sorted(line, key=lambda w: w["x0"])
        out.append((min(w["top"] for w in line), " ".join(w["text"] for w in line_sorted)))
    return out

def find_header_bottom_old(page):
    words = page.extract_words()
    header_words = [w for w in words if w["text"] in ("State", "Sector") and 60 < w["top"] < 100]
    if len(header_words) < 2:
        return None
    return max(w["bottom"] for w in header_words)

def extract_page_projects_old(page, header_bottom):
    words = page.extract_words()
    data_words = [w for w in words if w["top"] > header_bottom]
    if not data_words:
        return []
    anchors = sorted(
        [w for w in data_words if col_for_x(w["x0"]) == "sl_no" and w["text"].isdigit()],
        key=lambda w: w["top"],
    )
    if not anchors:
        return []
    bands = []
    for i, a in enumerate(anchors):
        top = a["top"] - 1
        bottom = anchors[i + 1]["top"] - 1 if i + 1 < len(anchors) else page.height
        bands.append((a["text"], top, bottom))

    projects = []
    for sl_no, top, bottom in bands:
        band_words = [w for w in data_words if top <= w["top"] < bottom]
        cols = {name: [] for name, _, _ in COLUMNS}
        for w in band_words:
            c = col_for_x(w["x0"])
            if c:
                cols[c].append(w)
        cell = {name: "\n".join(t for _, t in group_lines(cols[name])) for name in cols}
        cell["sl_no"] = sl_no
        projects.append(cell)
    return projects

def extract_old_format(path):
    fname = os.path.basename(path)
    all_projects = []
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            hb = find_header_bottom_old(page)
            if hb is None:
                continue
            all_projects.extend(extract_page_projects_old(page, hb))

    if not all_projects:
        print(f"  WARNING: {fname} — no matching table structure found at all")
        return []

    segments, current, prev = [], [], None
    for p in all_projects:
        sl = int(p["sl_no"])
        if prev is not None and sl < prev:
            segments.append(current)
            current = []
        current.append(p)
        prev = sl
    if current:
        segments.append(current)
    target = max(segments, key=len)
    print(f"  {fname}: {len(segments)} table segment(s) found, sizes {[len(s) for s in segments]}")
    print(f"  -> using largest segment: {len(target)} rows")

    rows_out = []
    cur_state, cur_sector = "", ""
    for p in target:
        if p["state"]:
            cur_state = normalize_state(p["state"].replace("\n", " "))
        if p["sector"]:
            cur_sector = p["sector"].replace("\n", " ").strip()

        name, agency, code = parse_project_cell(p["name"])
        approval = p["approval"].replace("\n", " ").strip()
        doc_parts = p["commission"].split("\n")
        doc_o = doc_parts[0].strip() if len(doc_parts) > 0 else ""
        doc_r = doc_parts[1].strip("() ") if len(doc_parts) > 1 else ""
        cost_parts = p["cost"].split("\n")
        cost_o = cost_parts[0].strip() if len(cost_parts) > 0 else ""
        cost_r = cost_parts[1].strip("() ") if len(cost_parts) > 1 else ""
        exp = p["exp"].strip()
        prog = p["progress"].strip()

        rows_out.append([
            fname, "old", p["sl_no"], name, agency, code, cur_state,
            "", cur_sector,
            approval, "",
            doc_o, doc_r,
            cost_o, cost_r,
            exp, prog,
        ])
    return rows_out

def main():
    all_rows = []
    for fname in TARGET_FILES:
        path = os.path.join(INPUT_DIR, fname)
        if not os.path.exists(path):
            print(f"MISSING: {path} — check the filename/path")
            continue
        print(f"Processing {fname} ...")
        rows = extract_old_format(path)
        all_rows.extend(rows)

    os.makedirs(os.path.dirname(OUT_PATH), exist_ok=True)
    with open(OUT_PATH, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(all_rows)

    print(f"\nTotal rows: {len(all_rows)} -> {OUT_PATH}")

if __name__ == "__main__":
    main()

Processing FRApril2025.pdf ...
  FRApril2025.pdf: 2 table segment(s) found, sizes [210, 1670]
  -> using largest segment: 1670 rows
Processing FR_JUNE_2025.pdf ...
  FR_JUNE_2025.pdf: 2 table segment(s) found, sizes [189, 1595]
  -> using largest segment: 1595 rows
Processing FR_May2025.pdf ...
  FR_May2025.pdf: 34 table segment(s) found, sizes [206, 3, 68, 12, 3, 4, 9, 6, 4, 4, 2, 3, 2, 4, 3, 2, 1, 3, 2, 1, 4, 4, 1, 2, 2, 3, 2, 3, 4, 3, 2, 2, 3, 1]
  -> using largest segment: 206 rows

Total rows: 3471 -> dataset_pdfs/old_format_3files.csv


In [ ]:
import pdfplumber

path = "dataset_pdfs/dataset_pdfs/FR_May2025.pdf"

with pdfplumber.open(path) as pdf:
    for i, page in enumerate(pdf.pages):
        text = page.extract_text() or ""

        if "Project List: Ongoing Projects" in text or "Ongoing Projects as of" in text:
            print("PAGE:", i + 1)
            print(text[:1500])

Streaming output truncated to the last 5000 lines.
Table:-7. Project List: Ongoing Projects as of 31st May 2025
State Sector Sl No Project Name Date Date of Commissioning Cost Original Cumulative Physical
(Agency Name) of Original (Revised) Expenditure Progress
(Project Code) Approval (Revised) {Anticipated} in Rs. Crore (%)
(MM/YYYY) {Anticipated} in Rs. Crore
(MM/YYYY)
287 AMERA OC RCE 2-2015 3/2018 335.97 86.98 42.00
(SECL) (Mar-2026) (N.A.)
(N06000123) {3/2026} {335.97}
288 AMGAON OPENCAST PROJECT RCE 7-2015 3/2020 316.10 188.33 71.00
(SECL) (Mar-2026) (N.A.)
(N06000135) {3/2026} {316.10}
289 MAHAMAYA OPENCAST PROJECT 5-2018 3/2026 410.26 180.18 38.00
(SECL) (N.A.) (N.A.)
(N06000164) {3/2026} {410.26}
290 JHIRIA WEST OPENCAST PROJECT 1-2024 3/2029 366.94 33.37 41.00
(SECL) (N.A.) (N.A.)
(N06000166) {3/2029} {366.94}
291 BAROUD OC EXPANSION (3-10 MTY) 8-2020 3/2029 2,309.80 911.51 41.00
(SECL) (N.A.) (N.A.)
(N06000197) {3/2029} {2,309.80}
292 JAMPALI OCP RCE 5-2020 3/2024 278.64 209

In [ ]:
import pdfplumber
import csv
import re
import os

INPUT_DIR = "dataset_pdfs/dataset_pdfs"
OUT_PATH = "dataset_pdfs/old_format_3files.csv"

TARGET_FILES = [
    "FRApril2025.pdf",
    "FR_May2025.pdf",
    "FR_JUNE_2025.pdf"
]

EXPECTED_COUNTS = {
    "FRApril2025.pdf": 1670,
    "FR_May2025.pdf": 1637,
    "FR_JUNE_2025.pdf": 1595
}

HEADER = [
    "report_file", "format", "sl_no", "project_name", "agency", "project_code",
    "state", "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

INDIAN_STATES_UTS = [
    "ANDAMAN AND NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH",
    "ASSAM", "BIHAR", "CHANDIGARH", "CHHATTISGARH",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "DELHI", "GOA", "GUJARAT",
    "HARYANA", "HIMACHAL PRADESH", "JAMMU AND KASHMIR", "JHARKHAND",
    "KARNATAKA", "KERALA", "LADAKH", "LAKSHADWEEP", "MADHYA PRADESH",
    "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", "NAGALAND",
    "ODISHA", "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM",
    "TAMIL NADU", "TELANGANA", "TRIPURA", "UTTAR PRADESH",
    "UTTARAKHAND", "WEST BENGAL",
]


def parse_project_cell(cell):
    if not cell:
        return "", "", ""

    text = cell.replace("\n", " ").strip()

    # Project code: (N04000092)
    code_match = re.search(r"\((\w{5,12})\s*\)\s*$", text)

    code = code_match.group(1) if code_match else ""

    if code_match:
        text = text[:code_match.start()].strip()

    # Agency: (AAI), (NHPC), (KMDA), etc.
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)

    agency = agency_match.group(1).strip() if agency_match else ""

    if agency_match:
        text = text[:agency_match.start()].strip()

    return text, agency, code


def normalize_state(raw):
    if not raw:
        return ""

    raw = raw.replace("\n", " ").strip().upper()

    for state in INDIAN_STATES_UTS:
        if raw == state:
            return state

    return raw


def extract_old_format(path):

    fname = os.path.basename(path)
    expected_count = EXPECTED_COUNTS[fname]

    raw_projects = []
    expected_sl_no = 1

    with pdfplumber.open(path) as pdf:

        for page_num, page in enumerate(pdf.pages, start=1):

            text = page.extract_text() or ""

            # Only process Table 7
            if "Table:-7." not in text:
                continue

            table = page.extract_table()

            if not table:
                continue

            # Find the Table 7 header row
            header_index = None

            for i, row in enumerate(table[:6]):

                row_text = " ".join(
                    str(cell or "") for cell in row
                )

                if (
                    "Sl No" in row_text
                    and "Project Name" in row_text
                    and "Physical" in row_text
                ):
                    header_index = i
                    break

            if header_index is None:
                continue

            # Extract project rows
            for row in table[header_index + 1:]:

                if len(row) < 9:
                    continue

                sl_no = (row[2] or "").strip()

                # The reports use sequential Sl No: 1,2,3,...,1637
                if not sl_no.isdigit():
                    continue

                if int(sl_no) != expected_sl_no:
                    continue

                raw_projects.append(row)

                expected_sl_no += 1

                if expected_sl_no > expected_count:
                    break

            if expected_sl_no > expected_count:
                break

    print(f"  {fname}: extracted {len(raw_projects)} projects")

    if len(raw_projects) != expected_count:
        print(
            f"  WARNING: expected {expected_count}, "
            f"but found {len(raw_projects)}"
        )

    # Convert extracted table rows into final CSV format
    rows_out = []

    current_state = ""
    current_sector = ""

    for row in raw_projects:

        # Table columns:
        # 0 State
        # 1 Sector
        # 2 Sl No
        # 3 Project Name
        # 4 Approval
        # 5 Commissioning
        # 6 Cost
        # 7 Expenditure
        # 8 Progress

        state = (row[0] or "").strip()
        sector = (row[1] or "").strip()

        if state:
            current_state = normalize_state(state)

        if sector:
            current_sector = sector.replace("\n", " ").strip()

        sl_no = (row[2] or "").strip()

        # Project name + agency + code
        name, agency, code = parse_project_cell(row[3] or "")

        # Approval date
        approval = (row[4] or "").replace("\n", " ").strip()

        # Commissioning dates
        commission_parts = (row[5] or "").split("\n")

        target_doc = (
            commission_parts[0].strip()
            if len(commission_parts) > 0
            else ""
        )

        revised_doc = (
            commission_parts[1].strip("() ")
            if len(commission_parts) > 1
            else ""
        )

        # Cost
        cost_parts = (row[6] or "").split("\n")

        original_cost = (
            cost_parts[0].strip()
            if len(cost_parts) > 0
            else ""
        )

        revised_cost = (
            cost_parts[1].strip("() ")
            if len(cost_parts) > 1
            else ""
        )

        # Expenditure
        expenditure = (
            row[7] or ""
        ).replace("\n", " ").strip()

        # Progress
        progress = (
            row[8] or ""
        ).replace("\n", " ").strip()

        rows_out.append([
            fname,
            "old",
            sl_no,
            name,
            agency,
            code,
            current_state,
            "",
            current_sector,
            approval,
            "",
            target_doc,
            revised_doc,
            original_cost,
            revised_cost,
            expenditure,
            progress
        ])

    return rows_out


def main():

    all_rows = []

    for fname in TARGET_FILES:

        path = os.path.join(INPUT_DIR, fname)

        if not os.path.exists(path):
            print(f"MISSING: {path}")
            continue

        print(f"\nProcessing {fname} ...")

        rows = extract_old_format(path)

        all_rows.extend(rows)

    os.makedirs(
        os.path.dirname(OUT_PATH),
        exist_ok=True
    )

    with open(
        OUT_PATH,
        "w",
        newline="",
        encoding="utf-8"
    ) as f:

        writer = csv.writer(f)

        writer.writerow(HEADER)
        writer.writerows(all_rows)

    print("\n===================================")
    print(f"TOTAL ROWS: {len(all_rows)}")
    print(f"OUTPUT: {OUT_PATH}")
    print("===================================")


if __name__ == "__main__":
    main()


Processing FRApril2025.pdf ...
  FRApril2025.pdf: extracted 1670 projects

Processing FR_May2025.pdf ...
  FR_May2025.pdf: extracted 1637 projects

Processing FR_JUNE_2025.pdf ...
  FR_JUNE_2025.pdf: extracted 1595 projects

TOTAL ROWS: 4902
OUTPUT: dataset_pdfs/old_format_3files.csv


In [ ]:
import pandas as pd
df = pd.read_csv("dataset_pdfs/old_format_3files.csv")
print(df.shape)
print(df['state'].value_counts().head(10))
print(df[['project_name','agency','project_code','approval_date','target_doc']].sample(5))

FileNotFoundError: [Errno 2] No such file or directory: 'dataset_pdfs/old_format_3files.csv'

In [ ]:
import os

print("Current directory:", os.getcwd())

print("\nSearching for CSV...")
for root, dirs, files in os.walk("/content"):
    for file in files:
        if file == "old_format_3files.csv":
            print("FOUND:", os.path.join(root, file))

Current directory: /content

Searching for CSV...


In [ ]:
import os
import glob

print("Folders in /content:")
for x in os.listdir("/content"):
    print(x)

print("\nAll CSV files under /content:")
csvs = glob.glob("/content/**/*.csv", recursive=True)

if csvs:
    for x in csvs:
        print(x)
else:
    print("NO CSV FILES FOUND")

Folders in /content:
.config
sample_data

All CSV files under /content:
/content/sample_data/mnist_test.csv
/content/sample_data/mnist_train_small.csv
/content/sample_data/california_housing_train.csv
/content/sample_data/california_housing_test.csv


In [ ]:
# Check where the input PDFs actually are
import os
import glob

print("PDFs found:")
pdfs = glob.glob("/content/**/*.pdf", recursive=True)

for p in pdfs:
    print(p)

PDFs found:


In [ ]:
 !pip install pdfplumber -q
#
 from google.colab import files
 uploaded = files.upload()   # click "Choose files", pick your zip, WAIT for it to finish

 import zipfile
 zip_name = list(uploaded.keys())[0]
 with zipfile.ZipFile(zip_name, "r") as z:
     z.extractall("dataset_pdfs")


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 639.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.5/66.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 21.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 69.7 MB/s eta 0:00:00


Saving dataset_pdfs.zip to dataset_pdfs.zip


In [ ]:
import pdfplumber
import csv
import re
import os
import glob
import pandas as pd

# auto-detect the actual PDF folder (handles both flat and nested-zip cases)
candidates = [d for d in glob.glob("dataset_pdfs/**/", recursive=True)
              if glob.glob(os.path.join(d, "*.pdf"))]
INPUT_DIR = candidates[0] if candidates else "dataset_pdfs"
print("Using INPUT_DIR:", INPUT_DIR)

OUT_NEW = "dataset_pdfs/new_format.csv"
OUT_OLD = "dataset_pdfs/old_format.csv"
OUT_MASTER = "dataset_pdfs/master_dataset.csv"

HEADER = [
    "report_file", "format", "sl_no", "project_name", "agency", "project_code",
    "state", "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

INDIAN_STATES_UTS = [
    "ANDAMAN AND NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM",
    "BIHAR", "CHANDIGARH", "CHHATTISGARH",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "DELHI", "GOA", "GUJARAT",
    "HARYANA", "HIMACHAL PRADESH", "JAMMU AND KASHMIR", "JHARKHAND",
    "KARNATAKA", "KERALA", "LADAKH", "LAKSHADWEEP", "MADHYA PRADESH",
    "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA",
    "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM", "TAMIL NADU", "TELANGANA",
    "TRIPURA", "UTTAR PRADESH", "UTTARAKHAND", "WEST BENGAL",
]

def split_two(cell):
    if not cell:
        return "", ""
    parts = cell.split("\n")
    main = parts[0].strip()
    rev = parts[1].strip("() ") if len(parts) > 1 else ""
    return main, rev

def parse_project_cell(cell):
    if not cell:
        return "", "", ""
    text = cell.replace("\n", " ").strip()
    code_match = re.search(r"\((\w{5,12})\s*\)\s*$", text)
    code = code_match.group(1) if code_match else ""
    if code_match:
        text = text[:code_match.start()].strip()
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)
    agency = agency_match.group(1).strip() if agency_match else ""
    if agency_match:
        text = text[:agency_match.start()].strip()
    return text, agency, code

def normalize_state(raw):
    if not raw:
        return ""
    raw = raw.replace("\n", " ").strip().upper()
    for state in INDIAN_STATES_UTS:
        if raw == state:
            return state
    return raw

# ---------------- NEW format (SSRS, "All Ongoing Projects" title) ----------------

def find_new_format_pages(pdf):
    pages = []
    for i, page in enumerate(pdf.pages):
        txt = page.extract_text() or ""
        first_line = txt.split("\n")[0].strip() if txt else ""
        if first_line == "All Ongoing Projects":
            pages.append(i)
    return pages

def extract_new_format(path):
    fname = os.path.basename(path)
    rows_out = []
    current_ministry, current_sector = "", ""
    with pdfplumber.open(path) as pdf:
        target_pages = find_new_format_pages(pdf)
        if not target_pages:
            return None
        for pnum in target_pages:
            page = pdf.pages[pnum]
            tables = page.extract_tables()
            data_table = None
            for t in tables:
                if t and t[0] and t[0][0] == "Sl.No":
                    data_table = t
                    break
            if not data_table:
                continue
            for row in data_table[1:]:
                if not row or all(c in (None, "") for c in row):
                    continue
                sl_no = (row[0] or "").strip()
                if not sl_no and row[1] and all(c in (None, "") for c in row[2:]):
                    label = row[1].strip()
                    if label.lower().startswith(("ministry", "department")):
                        current_ministry, current_sector = label, ""
                    else:
                        current_sector = label
                    continue
                if not sl_no.isdigit():
                    continue
                project_name, agency, code = parse_project_cell(row[1])
                state = re.sub(r"\s*\n\s*", " ", (row[2] or "")).strip()
                approval, revised_start = split_two(row[3])
                target_doc, revised_doc = split_two(row[4])
                orig_cost, revised_cost = split_two(row[5])
                expenditure = (row[6] or "").strip()
                progress = (row[7] or "").strip()
                rows_out.append([
                    fname, "new", sl_no, project_name, agency, code, state,
                    current_ministry, current_sector, approval, revised_start,
                    target_doc, revised_doc, orig_cost, revised_cost, expenditure, progress,
                ])
    return rows_out

# ---------------- OLD format ("Table:-7." + strict sequential Sl No) ----------------

def extract_old_format(path, expected_count=None):
    fname = os.path.basename(path)
    raw_projects = []
    expected_sl_no = 1

    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            if "Table:-7." not in text:
                continue
            table = page.extract_table()
            if not table:
                continue
            header_index = None
            for i, row in enumerate(table[:6]):
                row_text = " ".join(str(c or "") for c in row)
                if "Sl No" in row_text and "Project Name" in row_text and "Physical" in row_text:
                    header_index = i
                    break
            if header_index is None:
                continue
            for row in table[header_index + 1:]:
                if len(row) < 9:
                    continue
                sl_no = (row[2] or "").strip()
                if not sl_no.isdigit() or int(sl_no) != expected_sl_no:
                    continue
                raw_projects.append(row)
                expected_sl_no += 1
                if expected_count and expected_sl_no > expected_count:
                    break
            if expected_count and expected_sl_no > expected_count:
                break

    if not raw_projects:
        return None  # not this format

    rows_out = []
    current_state, current_sector = "", ""
    for row in raw_projects:
        state, sector = (row[0] or "").strip(), (row[1] or "").strip()
        if state:
            current_state = normalize_state(state)
        if sector:
            current_sector = sector.replace("\n", " ").strip()
        sl_no = (row[2] or "").strip()
        name, agency, code = parse_project_cell(row[3] or "")
        approval = (row[4] or "").replace("\n", " ").strip()
        commission_parts = (row[5] or "").split("\n")
        target_doc = commission_parts[0].strip() if commission_parts else ""
        revised_doc = commission_parts[1].strip("() ") if len(commission_parts) > 1 else ""
        cost_parts = (row[6] or "").split("\n")
        original_cost = cost_parts[0].strip() if cost_parts else ""
        revised_cost = cost_parts[1].strip("() ") if len(cost_parts) > 1 else ""
        expenditure = (row[7] or "").replace("\n", " ").strip()
        progress = (row[8] or "").replace("\n", " ").strip()
        rows_out.append([
            fname, "old", sl_no, name, agency, code, current_state, "", current_sector,
            approval, "", target_doc, revised_doc, original_cost, revised_cost, expenditure, progress,
        ])
    return rows_out

# ---------------- driver ----------------

def main():
    pdf_files = sorted(
        p for p in glob.glob(os.path.join(INPUT_DIR, "**", "*"), recursive=True)
        if p.lower().endswith(".pdf")
    )
    print(f"Found {len(pdf_files)} PDFs\n")

    all_rows = []
    unrecognized = []
    for path in pdf_files:
        fname = os.path.basename(path)
        print(f"Processing {fname} ...")
        rows = extract_new_format(path)
        if rows is not None:
            print(f"  -> {len(rows)} rows [new format]")
            all_rows.extend(rows)
            continue
        rows = extract_old_format(path)
        if rows:
            print(f"  -> {len(rows)} rows [old format]")
            all_rows.extend(rows)
            continue
        print(f"  WARNING: could not extract — unrecognized layout")
        unrecognized.append(fname)

    with open(OUT_MASTER, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        writer.writerow(HEADER)
        writer.writerows(all_rows)

    print(f"\n{'='*50}")
    print(f"TOTAL ROWS: {len(all_rows)}")
    print(f"FILES PROCESSED: {len(pdf_files) - len(unrecognized)} / {len(pdf_files)}")
    if unrecognized:
        print(f"UNRECOGNIZED FILES (0 rows): {unrecognized}")
    print(f"SAVED -> {OUT_MASTER}")
    print(f"{'='*50}")

    # quick per-file breakdown
    df = pd.read_csv(OUT_MASTER)
    print("\nRows per file:")
    print(df["report_file"].value_counts())

main()


Using INPUT_DIR: dataset_pdfs/dataset_pdfs/
Found 15 PDFs

Processing FRApril2025.pdf ...
  -> 1670 rows [old format]
Processing FR_JUNE_2025.pdf ...
  -> 1595 rows [old format]
Processing FR_May2025.pdf ...
  -> 1637 rows [old format]
Processing FlashReport_April2026.pdf ...
  -> 1981 rows [new format]
Processing FlashReport_August_2025.pdf ...
  -> 800 rows [new format]
Processing FlashReport_December_2025.pdf ...
  -> 1345 rows [new format]
Processing FlashReport_February_2026.pdf ...
  -> 1897 rows [new format]
Processing FlashReport_January_2026.pdf ...
  -> 1605 rows [new format]
Processing FlashReport_July_2025.pdf ...
  -> 714 rows [new format]
Processing FlashReport_July_2026.pdf ...
  -> 1775 rows [new format]
Processing FlashReport_June_2026.pdf ...
  -> 1847 rows [new format]
Processing FlashReport_March_2026.pdf ...
  -> 1869 rows [new format]
Processing FlashReport_May2026.pdf ...
  -> 1987 rows [new format]
Processing FlashReport_November_2025.pdf ...
  -> 823 rows [new 

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving FlashReport_September_2025.pdf to FlashReport_September_2025.pdf


In [ ]:
import os
print(os.path.exists("FlashReport_September_2025.pdf"))

True


In [ ]:
import pdfplumber
import csv
import re
import os
import glob
import pandas as pd

# ============================================================
# EDIT THIS to your September file's actual name/path
# ============================================================
SEPT_PDF_PATH = "FlashReport_September_2025.pdf"  # <-- change if needed

MASTER_PATH = "dataset_pdfs/master_dataset.csv"
# If you saved a backup to Drive earlier, point here instead if the local one is gone:
# MASTER_PATH = "/content/drive/MyDrive/master_dataset.csv"

HEADER = [
    "report_file", "format", "sl_no", "project_name", "agency", "project_code",
    "state", "ministry", "sector",
    "approval_date", "revised_start_date",
    "target_doc", "revised_doc",
    "original_cost_cr", "revised_cost_cr",
    "cumulative_expenditure_cr", "physical_progress_pct",
]

INDIAN_STATES_UTS = [
    "ANDAMAN AND NICOBAR ISLANDS", "ANDHRA PRADESH", "ARUNACHAL PRADESH", "ASSAM",
    "BIHAR", "CHANDIGARH", "CHHATTISGARH",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU", "DELHI", "GOA", "GUJARAT",
    "HARYANA", "HIMACHAL PRADESH", "JAMMU AND KASHMIR", "JHARKHAND",
    "KARNATAKA", "KERALA", "LADAKH", "LAKSHADWEEP", "MADHYA PRADESH",
    "MAHARASHTRA", "MANIPUR", "MEGHALAYA", "MIZORAM", "NAGALAND", "ODISHA",
    "PUDUCHERRY", "PUNJAB", "RAJASTHAN", "SIKKIM", "TAMIL NADU", "TELANGANA",
    "TRIPURA", "UTTAR PRADESH", "UTTARAKHAND", "WEST BENGAL",
]

def split_two(cell):
    if not cell:
        return "", ""
    parts = cell.split("\n")
    main = parts[0].strip()
    rev = parts[1].strip("() ") if len(parts) > 1 else ""
    return main, rev

def parse_project_cell(cell):
    if not cell:
        return "", "", ""
    text = cell.replace("\n", " ").strip()
    code_match = re.search(r"\((\w{5,12})\s*\)\s*$", text)
    code = code_match.group(1) if code_match else ""
    if code_match:
        text = text[:code_match.start()].strip()
    agency_match = re.search(r"\(([^()]+)\)\s*$", text)
    agency = agency_match.group(1).strip() if agency_match else ""
    if agency_match:
        text = text[:agency_match.start()].strip()
    return text, agency, code

def normalize_state(raw):
    if not raw:
        return ""
    raw = raw.replace("\n", " ").strip().upper()
    for state in INDIAN_STATES_UTS:
        if raw == state:
            return state
    return raw

def find_new_format_pages(pdf):
    pages = []
    for i, page in enumerate(pdf.pages):
        txt = page.extract_text() or ""
        first_line = txt.split("\n")[0].strip() if txt else ""
        if first_line == "All Ongoing Projects":
            pages.append(i)
    return pages

def extract_new_format(path):
    fname = os.path.basename(path)
    rows_out = []
    current_ministry, current_sector = "", ""
    with pdfplumber.open(path) as pdf:
        target_pages = find_new_format_pages(pdf)
        if not target_pages:
            return None
        for pnum in target_pages:
            page = pdf.pages[pnum]
            tables = page.extract_tables()
            data_table = None
            for t in tables:
                if t and t[0] and t[0][0] == "Sl.No":
                    data_table = t
                    break
            if not data_table:
                continue
            for row in data_table[1:]:
                if not row or all(c in (None, "") for c in row):
                    continue
                sl_no = (row[0] or "").strip()
                if not sl_no and row[1] and all(c in (None, "") for c in row[2:]):
                    label = row[1].strip()
                    if label.lower().startswith(("ministry", "department")):
                        current_ministry, current_sector = label, ""
                    else:
                        current_sector = label
                    continue
                if not sl_no.isdigit():
                    continue
                project_name, agency, code = parse_project_cell(row[1])
                state = re.sub(r"\s*\n\s*", " ", (row[2] or "")).strip()
                approval, revised_start = split_two(row[3])
                target_doc, revised_doc = split_two(row[4])
                orig_cost, revised_cost = split_two(row[5])
                expenditure = (row[6] or "").strip()
                progress = (row[7] or "").strip()
                rows_out.append([
                    fname, "new", sl_no, project_name, agency, code, state,
                    current_ministry, current_sector, approval, revised_start,
                    target_doc, revised_doc, orig_cost, revised_cost, expenditure, progress,
                ])
    return rows_out

def extract_old_format(path, expected_count=None):
    fname = os.path.basename(path)
    raw_projects = []
    expected_sl_no = 1
    with pdfplumber.open(path) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            if "Table:-7." not in text:
                continue
            table = page.extract_table()
            if not table:
                continue
            header_index = None
            for i, row in enumerate(table[:6]):
                row_text = " ".join(str(c or "") for c in row)
                if "Sl No" in row_text and "Project Name" in row_text and "Physical" in row_text:
                    header_index = i
                    break
            if header_index is None:
                continue
            for row in table[header_index + 1:]:
                if len(row) < 9:
                    continue
                sl_no = (row[2] or "").strip()
                if not sl_no.isdigit() or int(sl_no) != expected_sl_no:
                    continue
                raw_projects.append(row)
                expected_sl_no += 1
                if expected_count and expected_sl_no > expected_count:
                    break
            if expected_count and expected_sl_no > expected_count:
                break
    if not raw_projects:
        return None
    rows_out = []
    current_state, current_sector = "", ""
    for row in raw_projects:
        state, sector = (row[0] or "").strip(), (row[1] or "").strip()
        if state:
            current_state = normalize_state(state)
        if sector:
            current_sector = sector.replace("\n", " ").strip()
        sl_no = (row[2] or "").strip()
        name, agency, code = parse_project_cell(row[3] or "")
        approval = (row[4] or "").replace("\n", " ").strip()
        commission_parts = (row[5] or "").split("\n")
        target_doc = commission_parts[0].strip() if commission_parts else ""
        revised_doc = commission_parts[1].strip("() ") if len(commission_parts) > 1 else ""
        cost_parts = (row[6] or "").split("\n")
        original_cost = cost_parts[0].strip() if cost_parts else ""
        revised_cost = cost_parts[1].strip("() ") if len(cost_parts) > 1 else ""
        expenditure = (row[7] or "").replace("\n", " ").strip()
        progress = (row[8] or "").replace("\n", " ").strip()
        rows_out.append([
            fname, "old", sl_no, name, agency, code, current_state, "", current_sector,
            approval, "", target_doc, revised_doc, original_cost, revised_cost, expenditure, progress,
        ])
    return rows_out

# ---------------- run just the September file ----------------

fname = os.path.basename(SEPT_PDF_PATH)
if not os.path.exists(SEPT_PDF_PATH):
    print(f"FILE NOT FOUND: {SEPT_PDF_PATH}")
    print("Fix SEPT_PDF_PATH at the top of this script to the real path, then re-run.")
else:
    rows = extract_new_format(SEPT_PDF_PATH)
    fmt = "new"
    if rows is None:
        rows = extract_old_format(SEPT_PDF_PATH)
        fmt = "old"

    if not rows:
        print(f"Could not extract {fname} — unrecognized layout. Send it back to me and I'll adjust the parser.")
    else:
        print(f"Extracted {len(rows)} rows from {fname} [{fmt} format]")

        # load existing master, drop any prior rows from this same file (avoids duplicates on re-run), append, save
        if os.path.exists(MASTER_PATH):
            master = pd.read_csv(MASTER_PATH)
            before = len(master)
            master = master[master["report_file"] != fname]
            dropped = before - len(master)
            if dropped:
                print(f"Removed {dropped} pre-existing rows for {fname} before re-adding (avoids duplicates)")
        else:
            master = pd.DataFrame(columns=HEADER)
            print(f"No existing master found at {MASTER_PATH} — starting a fresh one")

        new_df = pd.DataFrame(rows, columns=HEADER)
        master = pd.concat([master, new_df], ignore_index=True)
        master.to_csv(MASTER_PATH, index=False)

        print(f"\nMaster dataset now has {len(master)} rows across {master['report_file'].nunique()} files")
        print(master["report_file"].value_counts())

Extracted 772 rows from FlashReport_September_2025.pdf [new format]

Master dataset now has 23115 rows across 16 files
report_file
FlashReport_May2026.pdf           1987
FlashReport_April2026.pdf         1981
FlashReport_February_2026.pdf     1897
FlashReport_March_2026.pdf        1869
FlashReport_June_2026.pdf         1847
FlashReport_July_2026.pdf         1775
FRApril2025.pdf                   1670
FR_May2025.pdf                    1637
FlashReport_January_2026.pdf      1605
FR_JUNE_2025.pdf                  1595
FlashReport_December_2025.pdf     1345
FlashReport_November_2025.pdf      823
FlashReport_August_2025.pdf        800
FlashReport_October_2025.pdf       798
FlashReport_September_2025.pdf     772
FlashReport_July_2025.pdf          714
Name: count, dtype: int64


In [ ]:
from google.colab import files
files.download("dataset_pdfs/master_dataset.csv")

from google.colab import drive
drive.mount('/content/drive')
import shutil
shutil.copy("dataset_pdfs/master_dataset.csv", "/content/drive/MyDrive/master_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Mounted at /content/drive


'/content/drive/MyDrive/master_dataset.csv'

In [ ]:
import pandas as pd
df = pd.read_csv("dataset_pdfs/master_dataset.csv")

print(df['format'].value_counts())
print(df.isna().sum())
print(df.groupby('report_file')['state'].nunique())

format
new    18213
old     4902
Name: count, dtype: int64
report_file                     0
format                          0
sl_no                           0
project_name                    0
agency                        124
project_code                 7618
state                         357
ministry                     4902
sector                          0
approval_date                 601
revised_start_date           5637
target_doc                      0
revised_doc                    11
original_cost_cr                0
revised_cost_cr                 0
cumulative_expenditure_cr       0
physical_progress_pct           3
dtype: int64
report_file
FRApril2025.pdf                     3
FR_JUNE_2025.pdf                    2
FR_May2025.pdf                      3
FlashReport_April2026.pdf         107
FlashReport_August_2025.pdf       104
FlashReport_December_2025.pdf     101
FlashReport_February_2026.pdf     106
FlashReport_January_2026.pdf      106
FlashReport_July_2025.pdf         

In [ ]:
import re
import pdfplumber
import pandas as pd

def extract_clean_states_by_bbox(pdf_path, x0_floor=14.0, x1_ceil=90.0):
    """
    Extracts (sl_no, state) pairs from old-format Flash Reports using word bboxes.
    x0_floor accounts for the data text starting left of the header word 'State'.
    """
    records = []
    current_state = None

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            words = page.extract_words()
            if not words:
                continue

            # Group words by rounded top coordinate into visual lines
            lines = {}
            for w in sorted(words, key=lambda x: (x['top'], x['x0'])):
                line_key = round(w['top'] / 3.0) * 3
                lines.setdefault(line_key, []).append(w)

            for _, line_words in sorted(lines.items()):
                line_words = sorted(line_words, key=lambda w: w['x0'])
                first_word = line_words[0]['text'].strip()

                # State header rows start near the left margin without a leading sl_no
                if line_words[0]['x0'] <= x1_ceil:
                    # Check if the line begins with an integer sl_no
                    if not re.match(r'^\d+$', first_word):
                        candidate_text = " ".join(
                            w['text'] for w in line_words if w['x0'] <= 250.0
                        ).strip()
                        # Ignore common non-state headers
                        if candidate_text and not any(h in candidate_text.upper() for h in ["STATE", "PROJECT", "SECTOR", "MINISTRY"]):
                            current_state = candidate_text
                    else:
                        sl_no = int(first_word)
                        records.append({"sl_no": sl_no, "state_clean": current_state})

    return pd.DataFrame(records)

def apply_state_patch(df_master, old_pdf_files):
    """
    Patches broken state rows in-place for specified old-format files.
    """
    df_patched = df_master.copy()

    for pdf_file in old_pdf_files:
        print(f"Extracting true states for {pdf_file}...")
        state_df = extract_clean_states_by_bbox(pdf_file)

        # Guard against duplicates if sl_no resets or repeats across sectors
        state_df = state_df.drop_duplicates(subset=['sl_no'])

        # Mask matching the target report
        mask = df_patched['report_file'] == pdf_file

        # Merge on sl_no
        merged = df_patched.loc[mask].merge(
            state_df[['sl_no', 'state_clean']],
            on='sl_no',
            how='left'
        )

        # Overwrite state with cleaned state
        df_patched.loc[mask, 'state'] = merged['state_clean'].values

    return df_patched

In [ ]:
old_files = ["FRApril2025.pdf", "FR_May2025.pdf", "FR_JUNE_2025.pdf"]
df_fixed = apply_state_patch(df, old_files)

# Quick Sanity Check
print("Unique states per old report after patch:")
print(df_fixed[df_fixed['format'] == 'old'].groupby('report_file')['state'].nunique())

print("\nMissing values in patched state column:")
print(df_fixed.loc[df_fixed['format'] == 'old', 'state'].isna().sum())

Extracting true states for FRApril2025.pdf...


FileNotFoundError: [Errno 2] No such file or directory: 'FRApril2025.pdf'

In [ ]:
import os

pdf_dir = "dataset_pdfs"
old_files = [
    os.path.join(pdf_dir, "FRApril2025.pdf"),
    os.path.join(pdf_dir, "FR_May2025.pdf"),
    os.path.join(pdf_dir, "FR_JUNE_2025.pdf"),
]

In [ ]:
def apply_state_patch(df_master, old_pdf_paths):
    df_patched = df_master.copy()

    for pdf_path in old_pdf_paths:
        file_name = os.path.basename(pdf_path)
        print(f"Extracting true states for {file_name} from {pdf_path}...")

        state_df = extract_clean_states_by_bbox(pdf_path)
        state_df = state_df.drop_duplicates(subset=['sl_no'])

        # Match using the base filename present in your DataFrame
        mask = df_patched['report_file'] == file_name

        merged = df_patched.loc[mask].merge(
            state_df[['sl_no', 'state_clean']],
            on='sl_no',
            how='left'
        )

        df_patched.loc[mask, 'state'] = merged['state_clean'].values

    return df_patched

# Run with paths
df_fixed = apply_state_patch(df, old_files)

# Sanity Check
print("Unique states per old report after patch:")
print(df_fixed[df_fixed['format'] == 'old'].groupby('report_file')['state'].nunique())

Extracting true states for FRApril2025.pdf from dataset_pdfs/FRApril2025.pdf...


FileNotFoundError: [Errno 2] No such file or directory: 'dataset_pdfs/FRApril2025.pdf'

In [ ]:
import os

print("Current working directory:", os.getcwd())
print("Files/folders in current dir:", os.listdir("."))

# Check specifically for dataset_pdfs
if os.path.exists("dataset_pdfs"):
    print("Files inside dataset_pdfs:", os.listdir("dataset_pdfs"))
else:
    print("Directory 'dataset_pdfs' was not found here.")

Current working directory: /content
Files/folders in current dir: ['.config', 'dataset_pdfs.zip', 'dataset_pdfs', 'FlashReport_September_2025.pdf', 'drive', 'sample_data']
Files inside dataset_pdfs: ['dataset_pdfs', 'master_dataset.csv']


In [ ]:
import os

# Point to the nested directory
pdf_dir = "dataset_pdfs/dataset_pdfs"

old_files = [
    os.path.join(pdf_dir, "FRApril2025.pdf"),
    os.path.join(pdf_dir, "FR_May2025.pdf"),
    os.path.join(pdf_dir, "FR_JUNE_2025.pdf"),
]

# Run the same patch logic as before
df_fixed = apply_state_patch(df, old_files)

# Quick Sanity Check
print("Unique states per old report after patch:")
print(df_fixed[df_fixed['format'] == 'old'].groupby('report_file')['state'].nunique())
print("\nMissing values in patched state column:")
print(df_fixed.loc[df_fixed['format'] == 'old', 'state'].isna().sum())

Extracting true states for FRApril2025.pdf from dataset_pdfs/dataset_pdfs/FRApril2025.pdf...
Extracting true states for FR_May2025.pdf from dataset_pdfs/dataset_pdfs/FR_May2025.pdf...
Extracting true states for FR_JUNE_2025.pdf from dataset_pdfs/dataset_pdfs/FR_JUNE_2025.pdf...
Unique states per old report after patch:
report_file
FRApril2025.pdf     1
FR_JUNE_2025.pdf    1
FR_May2025.pdf      1
Name: state, dtype: int64

Missing values in patched state column:
4803


In [ ]:
import os
import re
import pdfplumber
import pandas as pd

# Diagnostic inspection: Print text lines from pages with project data
pdf_sample = "dataset_pdfs/dataset_pdfs/FRApril2025.pdf"

with pdfplumber.open(pdf_sample) as pdf:
    for p_idx in range(min(15, len(pdf.pages))):
        text = pdf.pages[p_idx].extract_text() or ""
        lines = [line.strip() for line in text.splitlines() if line.strip()]
        # Check if page has project-like lines
        if any(re.match(r"^\d+\s+", l) for l in lines):
            print(f"--- Page {p_idx + 1} Sample Lines ---")
            for l in lines[:12]:
                print(f"  {l}")
            break

--- Page 7 Sample Lines ---
  MOSPI_ (April 2025) _FR_ Central Sector Projects cost Rs. 150Cr and above
  Table:-1. Overview of Ongoing Projects: Sector-wise Distribution
  Sl. No. Sector Projects Cost Cumulative
  Original Expenditure
  (Revised)* in Rs. Crore
  {Anticipated}
  in Rs. Crore
  1 CIVIL AVIATION 38 26,718.18 17,430.52
  (27,771.19)
  {30,473.90}
  2 COAL 113 188,127.06 66,702.32
  (192,671.10)


In [ ]:
import pdfplumber

pdf_path = "dataset_pdfs/dataset_pdfs/FRApril2025.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for idx, page in enumerate(pdf.pages):
        text = page.extract_text() or ""
        # Look for headers characteristic of the detailed project tables
        if "PROJECT-WISE" in text.upper() or "ANNEXURE" in text.upper() or "NAME OF THE PROJECT" in text.upper():
            print(f"--- Found Annexure Candidate on Page {idx + 1} ---")
            lines = [l.strip() for l in text.splitlines() if l.strip()]
            for l in lines[:15]:
                print(f"  {l}")
            break

In [ ]:
import pdfplumber

pdf_path = "dataset_pdfs/dataset_pdfs/FRApril2025.pdf"

with pdfplumber.open(pdf_path) as pdf:
    for idx, page in enumerate(pdf.pages):
        tables = page.extract_tables()
        for t in tables:
            # Look for a table with multiple columns and data rows
            if len(t) > 3 and len(t[0]) >= 8:
                print(f"Found project table on Page {idx + 1}")
                print(f"Column headers (row 0): {t[0]}")
                print(f"Row 1: {t[1]}")
                print(f"Row 2: {t[2]}")
                break
        else:
            continue
        break

Found project table on Page 18
Column headers (row 0): ['State', 'Sector', 'Sl No', 'Project Name\n(Agency Name)\n(Project Code)', 'Date\nof\nApproval\n(MM/YYYY)', 'Date of\nCommissioning\nOriginal\n(Revised)\n{Anticipated}\n(MM/YYYY)', 'Cost Original\n(Revised)\n{Anticipated}\nin Rs. Crore', 'Cumulative\nExpenditure\nin Rs. Crore', 'Physical\nProgress\n(%)']
Row 1: [None, 'CIVIL AVIATION', '1', 'CONSTRUCTION OF NEW GREEN FIELD\nAIRPORT AT HOLLONGI ITANAGAR\nARUNACHAL PRADESH\n(AAI)\n(N04000092)', '2-2019', '8/2022\n(5/2025)\n{5/2025}', '645.63\n(N.A.)\n{732.00}', '690.39', '99.5']
Row 2: [None, 'POWER', '2', 'SUBANSIRI LOWER H.E.P (8X250 MW)\n(NHPC)\n(NHPC)\n(180100221)', '9-2003', '9/2010\n(N.A.)\n{5/2026}', '6,285.33\n(N.A.)\n{26,075.54}', '24,022.52', '96.2']


In [ ]:
import pdfplumber

pdf_path = "dataset_pdfs/dataset_pdfs/FRApril2025.pdf"

with pdfplumber.open(pdf_path) as pdf:
    table = pdf.pages[17].extract_tables()[0] # page 18 (0-indexed 17)
    for i, row in enumerate(table[:10]):
        print(f"Row {i}: col0='{row[0]}' | sl_no='{row[2]}'")

Row 0: col0='State' | sl_no='Sl No'
Row 1: col0='None' | sl_no='1'
Row 2: col0='None' | sl_no='2'
Row 3: col0='None' | sl_no='3'
Row 4: col0='None' | sl_no='4'
Row 5: col0='None' | sl_no='5'
Row 6: col0='None' | sl_no='6'
Row 7: col0='None' | sl_no='7'
Row 8: col0='None' | sl_no='8'


In [ ]:
import pdfplumber

pdf_path = "dataset_pdfs/dataset_pdfs/FRApril2025.pdf"

with pdfplumber.open(pdf_path) as pdf:
    p = pdf.pages[17]  # page 18
    words = p.extract_words()
    left_words = [w for w in words if w['x0'] < 100]

    print(f"Total words on page: {len(words)}")
    print(f"Words with x0 < 100: {len(left_words)}\n")
    for w in sorted(left_words, key=lambda x: (x['top'], x['x0']))[:25]:
        print(f"text='{w['text']}' | x0={w['x0']:.1f}, x1={w['x1']:.1f}, top={w['top']:.1f}, bottom={w['bottom']:.1f}")

Total words on page: 272
Words with x0 < 100: 11

text='Sector' | x0=72.0, x1=92.2, top=88.8, bottom=96.7
text='State' | x0=16.1, x1=32.0, top=88.8, bottom=96.7
text='CIVIL' | x0=72.0, x1=88.6, top=162.2, bottom=170.1
text='AVIATION' | x0=90.4, x1=121.4, top=162.2, bottom=170.1
text='ARUNACHAL' | x0=16.1, x1=57.6, top=162.2, bottom=170.2
text='PRADESH' | x0=16.1, x1=47.8, top=172.8, bottom=180.7
text='POWER' | x0=72.0, x1=96.5, top=235.7, bottom=243.6
text='ROAD' | x0=72.0, x1=91.1, top=358.1, bottom=366.0
text='TRANSPORT' | x0=92.9, x1=133.3, top=358.1, bottom=366.0
text='AND' | x0=72.0, x1=86.1, top=368.6, bottom=376.6
text='HIGHWAYS' | x0=87.9, x1=123.6, top=368.6, bottom=376.6


In [ ]:
import os
import re
import pdfplumber
import pandas as pd

def extract_old_format_states(pdf_path):
    records = []
    current_state = None

    with pdfplumber.open(pdf_path) as pdf:
        for page_num, page in enumerate(pdf.pages, 1):
            words = page.extract_words()
            if not words:
                continue

            # Identify if page is a project annexure page (has 'State' at x0 ~ 16 and 'Sl No' / 'Sector')
            has_table_header = any(
                w['text'].upper() == 'STATE' and 10 <= w['x0'] <= 25 and w['top'] < 120
                for w in words
            )
            # Find serial number column (Sl No sits roughly between x0=100 and x0=150, or check integer labels)
            sl_words = [
                w for w in words
                if re.match(r'^\d+$', w['text'])
                and 100 <= w['x0'] <= 160
                and w['top'] > 100
            ]

            if not sl_words and not has_table_header:
                continue

            # 1. Collect all state text fragments on this page (x0 between 12 and 65, below headers)
            state_words = [
                w for w in words
                if 12 <= w['x0'] <= 65
                and w['top'] > 100
                and w['text'].upper() not in ['STATE', 'SL', 'NO', 'SECTOR']
            ]

            # Group state words by vertical proximity (within 15pt = same state name block)
            state_blocks = []
            if state_words:
                sorted_sw = sorted(state_words, key=lambda w: w['top'])
                curr_block = [sorted_sw[0]]

                for w in sorted_sw[1:]:
                    if w['top'] - curr_block[-1]['top'] <= 18.0:
                        curr_block.append(w)
                    else:
                        name = " ".join(item['text'] for item in curr_block).strip()
                        state_blocks.append({"name": name, "top": curr_block[0]['top']})
                        curr_block = [w]
                if curr_block:
                    name = " ".join(item['text'] for item in curr_block).strip()
                    state_blocks.append({"name": name, "top": curr_block[0]['top']})

            # 2. Walk through serial numbers and associate them with the active state
            # Combine state changes and project serial numbers chronologically by vertical top position
            events = []
            for sb in state_blocks:
                events.append(("state", sb['top'], sb['name']))
            for sw in sl_words:
                events.append(("sl_no", sw['top'], int(sw['text'])))

            # Sort by top coordinate down the page
            events.sort(key=lambda x: x[1])

            for ev_type, top_pos, val in events:
                if ev_type == "state":
                    # Filter out stray numbers or column artifacts
                    if not re.match(r'^\d+$', val) and len(val) > 2:
                        current_state = val
                elif ev_type == "sl_no":
                    records.append({
                        "sl_no": val,
                        "state_clean": current_state
                    })

    df_out = pd.DataFrame(records)
    if df_out.empty:
        return df_out
    return df_out.drop_duplicates(subset=['sl_no'], keep='first')

In [ ]:
pdf_dir = "dataset_pdfs/dataset_pdfs"
old_files = [
    os.path.join(pdf_dir, "FRApril2025.pdf"),
    os.path.join(pdf_dir, "FR_May2025.pdf"),
    os.path.join(pdf_dir, "FR_JUNE_2025.pdf"),
]

df_fixed = df.copy()

for pdf_path in old_files:
    fname = os.path.basename(pdf_path)
    state_map = extract_old_format_states(pdf_path)
    print(f"\n{fname}: extracted {len(state_map)} projects across {state_map['state_clean'].nunique()} unique states.")
    print("Sample states extracted:", state_map['state_clean'].dropna().unique()[:6])

    mask = df_fixed['report_file'] == fname
    merged = df_fixed.loc[mask].merge(
        state_map[['sl_no', 'state_clean']],
        on='sl_no',
        how='left'
    )
    df_fixed.loc[mask, 'state'] = merged['state_clean'].values

print("\n================ Verification ================")
print(df_fixed[df_fixed['format'] == 'old'].groupby('report_file')['state'].nunique())
print("\nTotal unmapped (null) states in old format:", df_fixed.loc[df_fixed['format'] == 'old', 'state'].isna().sum())


FRApril2025.pdf: extracted 1671 projects across 36 unique states.
Sample states extracted: ['CIVIL AVIATION' 'PETROLEUM' 'POWER' 'ROAD TRANSPORT HIGHWAYS' 'ASSAM'
 'MANIPUR']

FR_May2025.pdf: extracted 1638 projects across 37 unique states.
Sample states extracted: ['COAL' 'HEALTH AND FAMILY WELFARE' 'HOME AFFAIRS' 'PETROLEUM' 'POWER'
 'RAILWAYS']

FR_JUNE_2025.pdf: extracted 1595 projects across 36 unique states.
Sample states extracted: ['PETROLEUM' 'POWER' 'RAILWAYS' 'ROAD TRANSPORT HIGHWAYS' 'SOCIAL JUSTICE'
 'URBAN DEVELOPMENT']

================ Verification ================
report_file
FRApril2025.pdf     36
FR_JUNE_2025.pdf    36
FR_May2025.pdf      37
Name: state, dtype: int64

Total unmapped (null) states in old format: 0


In [ ]:
import os
import re
import pdfplumber
import pandas as pd

SECTOR_KEYWORDS = {
    'CIVIL AVIATION', 'COAL', 'FERTILIZERS', 'MINES', 'STEEL', 'PETROLEUM',
    'POWER', 'RAILWAYS', 'ROAD TRANSPORT', 'HIGHWAYS', 'SHIPPING', 'PORTS',
    'TELECOMMUNICATIONS', 'URBAN DEVELOPMENT', 'WATER RESOURCES', 'ATOMIC ENERGY',
    'DEFENCE', 'HEALTH', 'FAMILY WELFARE', 'HOME AFFAIRS', 'SOCIAL JUSTICE',
    'CHEMICALS', 'PETROCHEMICALS', 'HEAVY INDUSTRY', 'INFORMATION TECHNOLOGY'
}

def extract_old_format_states_strict(pdf_path):
    records = []
    current_state = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            if not words:
                continue

            # Identify serial numbers (sl_no)
            sl_words = [
                w for w in words
                if re.match(r'^\d+$', w['text'])
                and 100 <= w['x0'] <= 165
                and w['top'] > 100
            ]
            if not sl_words:
                continue

            # Isolate State column words: strict bounding box x0 in [12, 45] and x1 <= 68
            state_words = [
                w for w in words
                if 12.0 <= w['x0'] <= 45.0
                and w['x1'] <= 68.0
                and w['top'] > 100
                and w['text'].upper() not in ['STATE', 'SL', 'NO']
            ]

            # Group state words by vertical proximity
            state_blocks = []
            if state_words:
                sorted_sw = sorted(state_words, key=lambda w: w['top'])
                curr_block = [sorted_sw[0]]

                for w in sorted_sw[1:]:
                    if w['top'] - curr_block[-1]['top'] <= 16.0:
                        curr_block.append(w)
                    else:
                        name = " ".join(item['text'] for item in curr_block).strip().upper()
                        state_blocks.append({"name": name, "top": curr_block[0]['top']})
                        curr_block = [w]
                if curr_block:
                    name = " ".join(item['text'] for item in curr_block).strip().upper()
                    state_blocks.append({"name": name, "top": curr_block[0]['top']})

            # Interleave state changes and project serial numbers down the page
            events = []
            for sb in state_blocks:
                # Guard against sector bleed
                if not any(k in sb['name'] for k in SECTOR_KEYWORDS) and len(sb['name']) > 2:
                    events.append(("state", sb['top'], sb['name']))

            for sw in sl_words:
                events.append(("sl_no", sw['top'], int(sw['text'])))

            events.sort(key=lambda x: x[1])

            for ev_type, top_pos, val in events:
                if ev_type == "state":
                    current_state = val
                elif ev_type == "sl_no":
                    records.append({
                        "sl_no": val,
                        "state_clean": current_state
                    })

    df_out = pd.DataFrame(records)
    if df_out.empty:
        return df_out
    return df_out.drop_duplicates(subset=['sl_no'], keep='first')

In [ ]:
pdf_dir = "dataset_pdfs/dataset_pdfs"
old_files = [
    os.path.join(pdf_dir, "FRApril2025.pdf"),
    os.path.join(pdf_dir, "FR_May2025.pdf"),
    os.path.join(pdf_dir, "FR_JUNE_2025.pdf"),
]

df_fixed = df.copy()

for pdf_path in old_files:
    fname = os.path.basename(pdf_path)
    state_map = extract_old_format_states_strict(pdf_path)

    mask = df_fixed['report_file'] == fname
    merged = df_fixed.loc[mask].merge(
        state_map[['sl_no', 'state_clean']],
        on='sl_no',
        how='left'
    )
    df_fixed.loc[mask, 'state'] = merged['state_clean'].values

    print(f"\n{fname}: {state_map['state_clean'].nunique()} distinct states.")
    print("Unique values:", sorted(df_fixed.loc[mask, 'state'].dropna().unique()))

print("\n--- Summary ---")
print("Null states across old-format records:", df_fixed.loc[df_fixed['format'] == 'old', 'state'].isna().sum())


FRApril2025.pdf: 31 distinct states.
Unique values: ['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH', 'MADHYA PRADESH', 'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'MULTI', 'NAGALAND', 'ODISHA', 'PUNJAB', 'RAJASTHAN', 'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH', 'UTTARAKHAND', 'WEST BENGAL']

FR_May2025.pdf: 30 distinct states.
Unique values: ['ASSAM', 'BIHAR', 'CHHATTISGAR H', 'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH', 'MADHYA PRADESH', 'MAHARASHTR A', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'MULTI', 'NAGALAND', 'ODISHA', 'PUNJAB', 'RAJASTHAN', 'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH', 'UTTARAKHAN D', 'WEST BENGAL']

FR_JUNE_2025.pdf: 33 distinct states.
Unique values: ['ANDHRA PRADESH', 'ASSAM', 'BIHAR', 'DELHI', 'GOA', 'GUJARAT

In [ ]:
import os
import re
import pdfplumber
import pandas as pd

SECTOR_KEYWORDS = {
    'CIVIL AVIATION', 'COAL', 'FERTILIZERS', 'MINES', 'STEEL', 'PETROLEUM',
    'POWER', 'RAILWAYS', 'ROAD TRANSPORT', 'HIGHWAYS', 'SHIPPING', 'PORTS',
    'TELECOMMUNICATIONS', 'URBAN DEVELOPMENT', 'WATER RESOURCES', 'ATOMIC ENERGY',
    'DEFENCE', 'HEALTH', 'FAMILY WELFARE', 'HOME AFFAIRS', 'SOCIAL JUSTICE',
    'CHEMICALS', 'PETROCHEMICALS', 'HEAVY INDUSTRY', 'INFORMATION TECHNOLOGY'
}

def extract_old_format_states_strict(pdf_path):
    records = []
    current_state = None

    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            words = page.extract_words()
            if not words:
                continue

            # Identify serial numbers (sl_no)
            sl_words = [
                w for w in words
                if re.match(r'^\d+$', w['text'])
                and 100 <= w['x0'] <= 165
                and w['top'] > 100
            ]
            if not sl_words:
                continue

            # Isolate State column words: strict bounding box x0 in [12, 45] and x1 <= 68
            state_words = [
                w for w in words
                if 12.0 <= w['x0'] <= 45.0
                and w['x1'] <= 68.0
                and w['top'] > 100
                and w['text'].upper() not in ['STATE', 'SL', 'NO']
            ]

            # Group state words by vertical proximity
            state_blocks = []
            if state_words:
                sorted_sw = sorted(state_words, key=lambda w: w['top'])
                curr_block = [sorted_sw[0]]

                for w in sorted_sw[1:]:
                    if w['top'] - curr_block[-1]['top'] <= 16.0:
                        curr_block.append(w)
                    else:
                        name = " ".join(item['text'] for item in curr_block).strip().upper()
                        state_blocks.append({"name": name, "top": curr_block[0]['top']})
                        curr_block = [w]
                if curr_block:
                    name = " ".join(item['text'] for item in curr_block).strip().upper()
                    state_blocks.append({"name": name, "top": curr_block[0]['top']})

            # Interleave state changes and project serial numbers down the page
            events = []
            for sb in state_blocks:
                # Guard against sector bleed
                if not any(k in sb['name'] for k in SECTOR_KEYWORDS) and len(sb['name']) > 2:
                    events.append(("state", sb['top'], sb['name']))

            for sw in sl_words:
                events.append(("sl_no", sw['top'], int(sw['text'])))

            events.sort(key=lambda x: x[1])

            for ev_type, top_pos, val in events:
                if ev_type == "state":
                    current_state = val
                elif ev_type == "sl_no":
                    records.append({
                        "sl_no": val,
                        "state_clean": current_state
                    })

    df_out = pd.DataFrame(records)
    if df_out.empty:
        return df_out
    return df_out.drop_duplicates(subset=['sl_no'], keep='first')

In [ ]:
pdf_dir = "dataset_pdfs/dataset_pdfs"
old_files = [
    os.path.join(pdf_dir, "FRApril2025.pdf"),
    os.path.join(pdf_dir, "FR_May2025.pdf"),
    os.path.join(pdf_dir, "FR_JUNE_2025.pdf"),
]

df_fixed = df.copy()

for pdf_path in old_files:
    fname = os.path.basename(pdf_path)
    state_map = extract_old_format_states_strict(pdf_path)

    mask = df_fixed['report_file'] == fname
    merged = df_fixed.loc[mask].merge(
        state_map[['sl_no', 'state_clean']],
        on='sl_no',
        how='left'
    )
    df_fixed.loc[mask, 'state'] = merged['state_clean'].values

    print(f"\n{fname}: {state_map['state_clean'].nunique()} distinct states.")
    print("Unique values:", sorted(df_fixed.loc[mask, 'state'].dropna().unique()))

print("\n--- Summary ---")
print("Null states across old-format records:", df_fixed.loc[df_fixed['format'] == 'old', 'state'].isna().sum())


FRApril2025.pdf: 31 distinct states.
Unique values: ['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH', 'MADHYA PRADESH', 'MAHARASHTRA', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'MULTI', 'NAGALAND', 'ODISHA', 'PUNJAB', 'RAJASTHAN', 'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH', 'UTTARAKHAND', 'WEST BENGAL']

FR_May2025.pdf: 30 distinct states.
Unique values: ['ASSAM', 'BIHAR', 'CHHATTISGAR H', 'DELHI', 'GOA', 'GUJARAT', 'HARYANA', 'HIMACHAL PRADESH', 'JAMMU AND KASHMIR', 'JHARKHAND', 'KARNATAKA', 'KERALA', 'LADAKH', 'MADHYA PRADESH', 'MAHARASHTR A', 'MANIPUR', 'MEGHALAYA', 'MIZORAM', 'MULTI', 'NAGALAND', 'ODISHA', 'PUNJAB', 'RAJASTHAN', 'SIKKIM', 'TAMIL NADU', 'TELANGANA', 'TRIPURA', 'UTTAR PRADESH', 'UTTARAKHAN D', 'WEST BENGAL']

FR_JUNE_2025.pdf: 33 distinct states.
Unique values: ['ANDHRA PRADESH', 'ASSAM', 'BIHAR', 'DELHI', 'GOA', 'GUJARAT

In [ ]:
import numpy as np

CLEANUP_MAP = {
    'CHHATTISGAR H': 'CHHATTISGARH',
    'MAHARASHTR A': 'MAHARASHTRA',
    'UTTARAKHAN D': 'UTTARAKHAND',
    'MULTI': 'MULTI STATE',
    'UTTAR': 'UTTAR PRADESH',
    'SOCIAL': np.nan,
    'URBAN': np.nan,
}

# 1. Clean noisy tokens and sector bleed
df_fixed['state'] = df_fixed['state'].replace(CLEANUP_MAP)

# 2. Fill sector-bleed gaps and unmapped lead rows per report
df_fixed['state'] = (
    df_fixed.groupby('report_file')['state']
    .ffill()
    .bfill()
)

In [ ]:
# Strip stray whitespace
df_fixed['state'] = df_fixed['state'].astype(str).str.strip().str.upper()

# Canonical harmonization across old and new formats
CANONICAL_STATE_MAP = {
    'ANDAMAN & NICOBAR': 'ANDAMAN AND NICOBAR ISLANDS',
    'ANDAMAN AND NICOBAR': 'ANDAMAN AND NICOBAR ISLANDS',
    'D & N HAVELI': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'DAMAN & DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU',
    'DELHI/NCR': 'DELHI',
    'NCT OF DELHI': 'DELHI',
    'JAMMU & KASHMIR': 'JAMMU AND KASHMIR',
    'ORISSA': 'ODISHA',
    'PONDICHERRY': 'PUDUCHERRY',
    'MULTI-STATE': 'MULTI STATE',
    'MULTI': 'MULTI STATE',
}

df_fixed['state'] = df_fixed['state'].replace(CANONICAL_STATE_MAP)

In [ ]:
print("--- Old Format Unique States ---")
print(df_fixed[df_fixed['format'] == 'old'].groupby('report_file')['state'].nunique())

print("\n--- Remaining Nulls Across Full Dataset ---")
print(df_fixed['state'].isna().sum())

print("\n--- Cross-Format Overlap Check ---")
old_states = set(df_fixed[df_fixed['format'] == 'old']['state'].unique())
new_states = set(df_fixed[df_fixed['format'] == 'new']['state'].unique())

diff = old_states - new_states
print(f"States in old format not present in new format: {diff if diff else 'None (Clean Match)'}")

--- Old Format Unique States ---
report_file
FRApril2025.pdf     31
FR_JUNE_2025.pdf    30
FR_May2025.pdf      30
Name: state, dtype: int64

--- Remaining Nulls Across Full Dataset ---
0

--- Cross-Format Overlap Check ---
States in old format not present in new format: {'MULTI STATE'}


In [ ]:
new_states = sorted(df_fixed[df_fixed['format'] == 'new']['state'].dropna().unique())
print([s for s in new_states if any(k in s for k in ['MULTI', 'ALL', 'CENTRAL', 'INTER'])])

['MULTI-STATES (ANDHRA PRADESH, BIHAR, CHHATTISGARH, JHARKHAND, MADHYA PRADESH, MAHARASHTRA, ODISHA, TELANGANA, UTTAR PRADESH, WEST BENGAL)', 'MULTI-STATES (ANDHRA PRADESH, BIHAR, DELHI, GUJARAT, JHARKHAND, KARNATAKA, KERALA, MADHYA PRADESH, ODISHA, TAMIL NADU, TELANGANA, UTTAR PRADESH, WEST BENGAL)', 'MULTI-STATES (ANDHRA PRADESH, CHHATTISGARH, JHARKHAND, MAHARASHTRA, ODISHA)', 'MULTI-STATES (ANDHRA PRADESH, CHHATTISGARH, ODISHA)', 'MULTI-STATES (ANDHRA PRADESH, KARNATAKA)', 'MULTI-STATES (ANDHRA PRADESH, MAHARASHTRA, TELANGANA)', 'MULTI-STATES (ANDHRA PRADESH, ODISHA)', 'MULTI-STATES (ANDHRA PRADESH, ODISHA, TELANGANA)', 'MULTI-STATES (ANDHRA PRADESH, TAMIL NADU)', 'MULTI-STATES (ANDHRA PRADESH, TELANGANA)', 'MULTI-STATES (ARUNACHAL PRADESH, ASSAM)', 'MULTI-STATES (ARUNACHAL PRADESH, ASSAM, MANIPUR, MEGHALAYA, MIZORAM, NAGALAND, SIKKIM, TRIPURA, WEST BENGAL)', 'MULTI-STATES (ARUNACHAL PRADESH, GUJARAT, RAJASTHAN)', 'MULTI-STATES (ARUNACHAL PRADESH, SIKKIM)', 'MULTI-STATES (ASSAM, BIH

In [ ]:
import re
import pandas as pd

def harmonize_multi_states(df):
    df_clean = df.copy()

    # 1. Normalize whitespace & casing
    df_clean['state'] = df_clean['state'].astype(str).str.strip().str.upper()

    # 2. Extract list of constituent states if present
    def extract_states(val):
        m = re.search(r'MULTI-STATES?\s*\((.*?)\)', val)
        if m:
            return [s.strip() for s in m.group(1).split(',') if s.strip()]
        if val in ['MULTI STATE', 'MULTI-STATE', 'MULTI']:
            return []
        return [val]

    df_clean['participating_states'] = df_clean['state'].apply(extract_states)

    # 3. Create boolean multi-state flag
    df_clean['is_multi_state'] = (
        df_clean['state'].str.startswith('MULTI') |
        (df_clean['participating_states'].apply(len) > 1)
    )

    # 4. Standardized high-level state column for models & aggregations
    df_clean['state_standardized'] = df_clean['state'].apply(
        lambda x: 'MULTI-STATE' if x.startswith('MULTI') else x
    )

    return df_clean

# Apply harmonization
df_final = harmonize_multi_states(df_fixed)

In [ ]:
# Check alignment across formats
old_states = set(df_final[df_final['format'] == 'old']['state_standardized'].unique())
new_states = set(df_final[df_final['format'] == 'new']['state_standardized'].unique())

print("States in Old but not New:", old_states - new_states)
print("States in New but not Old:", new_states - old_states)

print("\nMulti-state projects breakdown:")
print(df_final.groupby(['format', 'is_multi_state'])['sl_no'].count())

# Export clean production dataset
df_final.to_csv("dataset_pdfs/master_dataset_clean.csv", index=False)

States in Old but not New: set()
States in New but not Old: {'PAN INDIA', 'ANDAMAN AND NICOBAR ISLANDS', 'OFFSHORE', 'DADRA & NAGAR HAVELI AND DAMAN & DIU', 'CHANDIGARH', 'PUDUCHERRY'}

Multi-state projects breakdown:
format  is_multi_state
new     False             16250
        True               1963
old     False              4753
        True                149
Name: sl_no, dtype: int64


In [ ]:
# Unify ampersand variation
df_final['state_standardized'] = df_final['state_standardized'].replace({
    'DADRA & NAGAR HAVELI AND DAMAN & DIU': 'DADRA AND NAGAR HAVELI AND DAMAN AND DIU'
})

In [ ]:
new_only_candidates = ['PAN INDIA', 'OFFSHORE', 'CHANDIGARH', 'PUDUCHERRY', 'ANDAMAN AND NICOBAR ISLANDS']

print("Row counts for candidate states in 'new' format:")
print(df_final[df_final['state_standardized'].isin(new_only_candidates)]['state_standardized'].value_counts())

Row counts for candidate states in 'new' format:
state_standardized
OFFSHORE                       99
PAN INDIA                      90
ANDAMAN AND NICOBAR ISLANDS    29
PUDUCHERRY                     12
CHANDIGARH                      1
Name: count, dtype: int64


In [ ]:
# Overwrite your original master file
df_final.to_csv("dataset_pdfs/master_dataset.csv", index=False)
print("Updated master_dataset.csv successfully in-place.")

Updated master_dataset.csv successfully in-place.


In [ ]:
# Overwrite your original master file
df_final.to_csv("dataset_pdfs/master_dataset.csv", index=False)
print("Updated master_dataset.csv successfully in-place.")

Updated master_dataset.csv successfully in-place.


In [ ]:
from google.colab import files

files.download("dataset_pdfs/master_dataset.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>